In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.layers import LSTM,Bidirectional,GRU
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import to_categorical
import datetime
import io
import itertools
# import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold
from sklearn.metrics import classification_report

import sys
import os
# Obtener la ruta del directorio actual
os.chdir('..')
current_dir = os.getcwd()
print(current_dir)

# Construir la ruta relativa al directorio que quieres agregar
relative_dir = os.path.join(current_dir, 'mis_pkgs/')

# Agregar la ruta relativa al sys.path
sys.path.insert(0, relative_dir)

from MIOPATIA_db import DB_management as db 


2024-07-08 10:22:43.321555: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-08 10:22:43.321610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-08 10:22:43.323038: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-08 10:22:43.333237: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-08 10:22:44.182834: W tensorflow/compiler/tf2

/home/rgadea/nuevas_investigaciones_alimentos_2024


In [2]:
numero_muestras=201
numero_clases=2
entrada=[9,10]
numero_entradas =2
numero_epochs=2000

Voy a quedarme con los 50 atunes P1 para obtener conjunto de training y validacion

In [3]:
filename = "COPIA_PANDAS/medidas_agilent_2023_y_2024_201_puntos_clasificados.hdf"
with pd.HDFStore(filename,complib="zlib",complevel=4) as hdf_db:
    pre_p_e1  = hdf_db.get('data/pollos_estado')
    pre_p_e1 = pre_p_e1.loc[pre_p_e1['Pollo'] != 0]
    # p_e =pre_p_e1.drop_duplicates(subset = ['Pollo', 'Medida'],  keep = 'last').reset_index(drop = True)
    t    = hdf_db.get('data/tabla')
    X_train=np.zeros((pre_p_e1.shape[0],numero_muestras,numero_entradas))
    y_train=np.zeros((pre_p_e1.shape[0],1))
    x=0
    for index, row in pre_p_e1.iterrows():   # El primer registro no se toma en cuenta porque es basura
        Primero = int(row['Primero'])
        Ultimo  = int(row['Ultimo'])
        estado  = int(row['Estado'])
        #print(Primero)
        #print(Ultimo)
        #print(estado)
        if numero_clases==2:
            if estado == 0 or estado== 1:
                target = 0
            else:
                target = 1
        else:
            target=estado
        pepito=np.array(t.iloc[Primero:Ultimo+1])
        # #print(pepito.shape)
        X_train[x]=pepito[:,entrada]
        #print(X_train[x][0:4,:])       
        y_train[x]=target
        y_train_to_categorical = to_categorical(y_train)
        y_train_to_categorical = y_train
        x=x+1



X_train_filtrado = X_train
#y_train_filtrado = y_train
y_train_filtrado = y_train_to_categorical


scaler = MinMaxScaler(feature_range=(0, 1))
#scaler = StandardScaler()



#data1=np.concatenate((X_train_filtrado,X_test_filtrado1),axis=0) 

data_2d = X_train_filtrado.reshape(-1, X_train_filtrado.shape[-1])
normalized_data_2d = scaler.fit_transform(data_2d)



X_train_Normalizado=normalized_data_2d.reshape(X_train_filtrado.shape)
y_train_Normalizado=y_train_filtrado # los valores ya estaban normalizados
print(data_2d.shape)
print(X_train_Normalizado.shape)
print(y_train_Normalizado.shape)

inputs=X_train_Normalizado.reshape(X_train_Normalizado.shape[0],-1)
targets=y_train_Normalizado

print(inputs.shape)
print(targets.shape)



(38994, 2)
(194, 201, 2)
(194, 1)
(194, 402)
(194, 1)


Vamos a hacer los conjuntos de entrenamiento validacion y test

In [4]:
factor_aprendizaje=0.001
dimension_LSTM=50
dimension_dense1=200
dimension_dense2=20
algoritmo='SGD'
epochs=8000
supermax=8*4
lossfunction='binary_crossentropy'
def create_model():

    model = Sequential()
    model.add(GRU(dimension_LSTM, return_sequences=True,recurrent_regularizer='L2',input_shape=(numero_muestras, numero_entradas)))
    model.add(Flatten())  
    #model.add(GRU(50, return_sequences=True))
    #model.add(GRU(50, return_sequences=False, recurrent_regularizer='L2'))
    model.add(Dense(dimension_dense1, activation='tanh', kernel_regularizer='L2'))
    model.add(Dense(dimension_dense2, activation='tanh'))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss=lossfunction, optimizer=algoritmo, metrics=['accuracy',
                           #   tf.keras.metrics.Recall(class_id=0),
                           #   tf.keras.metrics.Recall(class_id=1) #,
                              #tfa.metrics.F1Score(num_classes=numero_clases,average='macro', threshold=0.5)
                              ])
    model.optimizer.lr=(factor_aprendizaje)
    return model



In [5]:

experimento="LOMOS_Agilent_5clases_GRU1_{}_dense1_{}_dense2_{}_loss_{}_lr_{}_algoritmo_{}".format(dimension_LSTM,dimension_dense1,dimension_dense2,lossfunction,factor_aprendizaje,algoritmo)
logdir="./logs/defs/{}_{}".format(experimento,datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=logdir, histogram_freq=1)
file_writer_cm = tf.summary.create_file_writer(logdir + '/cm')


2024-07-08 10:22:47.253576: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-08 10:22:47.296487: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-08 10:22:47.296582: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-08 10:22:47.299850: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-08 10:22:47.299947: I external/local_xla/xla/stream_executor

In [6]:
if numero_clases==2:
    class_names=['Buenos', 'Malos']
else:
    class_names=['A', 'B+', 'B', 'B-','C']

In [7]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.95,
    patience=50,
    min_lr=0.0001
)

In [8]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.95,
    patience=500,
    min_lr=0.0001
)
early_stop=tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=0, patience=2000, verbose=2, mode='auto', baseline=None, restore_best_weights=True)
# Define the K-fold Cross Validator
kfold = KFold(n_splits=10, shuffle=True)
# Define per-fold score containers
acc_per_fold = []
loss_per_fold = []
sensibilidad_YAKE_per_fold=[]
sensibilidad_no_YAKE_per_fold=[]
# K-fold Cross Validation model evaluation
fold_no = 1
for train, test in kfold.split(inputs, targets):
    model=create_model()
     # Generate a print
    print('------------------------------------------------------------------------')
    print(f'Training for fold {fold_no} ...')
    inputs_good=inputs.reshape(X_train_filtrado.shape)
    # Fit data to model
    history = model.fit(inputs_good[train], targets[train],
              batch_size=20,
              epochs=8,
              callbacks=[early_stop,lr_callback],
              validation_data=(inputs_good[test],targets[test])
              )
    if numero_clases==2:
        target_names = ['Buenos', 'Malos']
    else:   
        target_names = ['A', 'B+', 'B', 'B-','C']
    y_pred = model.predict(inputs_good[test])
 
    print(y_pred)
    print(y_pred2)
    # y_test_def2=np.argmax(targets[test],axis=1)
    report_dict = classification_report(targets[test], y_pred2, target_names=target_names, digits=4, output_dict=True)

    print(report_dict)
    # Generate generalization metrics
    scores = model.evaluate(inputs_good[test], targets[test], verbose=0)
    print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
    # Extract recall values
    recall_buenos = report_dict['Buenos']['recall']
    recall_malos = report_dict['Malos']['recall']
    print(f"Recall for Buenos: {recall_buenos}")
    print(f"Recall for Malos: {recall_malos}")
   # print(scores[4])
    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])
    sensibilidad_YAKE_per_fold.append(recall_buenos * 100)
    sensibilidad_no_YAKE_per_fold.append(recall_malos * 100)
    # Increase fold number
    fold_no = fold_no + 1

2024-07-08 10:22:47.910206: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


------------------------------------------------------------------------
Training for fold 1 ...
Epoch 1/8


2024-07-08 10:22:49.613309: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-07-08 10:22:49.931820: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f5510617290 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-07-08 10:22:49.931860: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1070 Ti, Compute Capability 6.1
2024-07-08 10:22:49.936596: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1720426969.972462  970965 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


9/9 [==============================] - 2s 66ms/step - loss: 5.1629 - accuracy: 0.5747 - val_loss: 3.7572 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 2/8
9/9 [==============================] - 0s 23ms/step - loss: 2.9167 - accuracy: 0.5747 - val_loss: 2.2496 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 3/8
9/9 [==============================] - 0s 26ms/step - loss: 1.8657 - accuracy: 0.5747 - val_loss: 1.5686 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 4/8
9/9 [==============================] - 0s 25ms/step - loss: 1.3372 - accuracy: 0.5747 - val_loss: 1.2047 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 5/8
9/9 [==============================] - 0s 33ms/step - loss: 1.2203 - accuracy: 0.5747 - val_loss: 1.0521 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 6/8
9/9 [==============================] - 0s 26ms/step - loss: 0.9577 - accuracy: 0.5747 - val_loss: 0.9427 - val_accuracy: 0.4500 - lr: 0.0010
Epoch 7/8
9/9 [==============================] - 0s 29ms/step - loss: 0.8658 - accuracy: 0.5747 - val_lo

/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


------------------------------------------------------------------------
Training for fold 9 ...
Epoch 1/8
9/9 [==============================] - 2s 65ms/step - loss: 4.3220 - accuracy: 0.4514 - val_loss: 2.8804 - val_accuracy: 0.4211 - lr: 0.0010
Epoch 2/8
9/9 [==============================] - 0s 28ms/step - loss: 2.3089 - accuracy: 0.5771 - val_loss: 1.8184 - val_accuracy: 0.4211 - lr: 0.0010
Epoch 3/8
9/9 [==============================] - 0s 34ms/step - loss: 1.5099 - accuracy: 0.6057 - val_loss: 1.3440 - val_accuracy: 0.5789 - lr: 0.0010
Epoch 4/8
9/9 [==============================] - 0s 26ms/step - loss: 1.2333 - accuracy: 0.4343 - val_loss: 1.0652 - val_accuracy: 0.4211 - lr: 0.0010
Epoch 5/8
9/9 [==============================] - 0s 28ms/step - loss: 0.9932 - accuracy: 0.5771 - val_loss: 0.9918 - val_accuracy: 0.4211 - lr: 0.0010
Epoch 6/8
9/9 [==============================] - 0s 29ms/step - loss: 0.8951 - accuracy: 0.5543 - val_loss: 1.0147 - val_accuracy: 0.4211 - lr: 0.00

/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


------------------------------------------------------------------------
Training for fold 10 ...
Epoch 1/8
9/9 [==============================] - 3s 69ms/step - loss: 4.2249 - accuracy: 0.5657 - val_loss: 2.8527 - val_accuracy: 0.5263 - lr: 0.0010
Epoch 2/8
9/9 [==============================] - 0s 35ms/step - loss: 2.2975 - accuracy: 0.5657 - val_loss: 1.7381 - val_accuracy: 0.5263 - lr: 0.0010
Epoch 3/8
9/9 [==============================] - 0s 30ms/step - loss: 1.4977 - accuracy: 0.5657 - val_loss: 1.2543 - val_accuracy: 0.5263 - lr: 0.0010
Epoch 4/8
9/9 [==============================] - 0s 28ms/step - loss: 1.1353 - accuracy: 0.5657 - val_loss: 1.0921 - val_accuracy: 0.4737 - lr: 0.0010
Epoch 5/8
9/9 [==============================] - 0s 33ms/step - loss: 1.0044 - accuracy: 0.5429 - val_loss: 0.9339 - val_accuracy: 0.5263 - lr: 0.0010
Epoch 6/8
9/9 [==============================] - 0s 26ms/step - loss: 0.8973 - accuracy: 0.5657 - val_loss: 0.8653 - val_accuracy: 0.5263 - lr: 0.0

/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/rgadea/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(acc_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Loss: {loss_per_fold[i]} - Accuracy: {acc_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Accuracy: {np.mean(acc_per_fold)} (+- {np.std(acc_per_fold)})')
print(f'> Loss: {np.mean(loss_per_fold)}')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Loss: 0.8006418347358704 - Accuracy: 55.000001192092896%
------------------------------------------------------------------------
> Fold 2 - Loss: 0.8136581182479858 - Accuracy: 50.0%
------------------------------------------------------------------------
> Fold 3 - Loss: 0.8389662504196167 - Accuracy: 50.0%
------------------------------------------------------------------------
> Fold 4 - Loss: 0.7623468041419983 - Accuracy: 69.9999988079071%
------------------------------------------------------------------------
> Fold 5 - Loss: 0.8209259510040283 - Accuracy: 52.63158082962036%
------------------------------------------------------------------------
> Fold 6 - Loss: 0.7708437442779541 - Accuracy: 63.15789222717285%
------------------------------------------------------------------------
> Fold 7 - Loss: 0.789076